In [7]:
import json
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

with open("/Users/nischayverma/Downloads/ord_dataset-00005539a1e04c809a9a78647bea649c.pb.json", "r") as file:
    data = json.load(file)

rows = []
for reaction in data:
    reactants = " .".join(
        [c["identifiers"][0]["value"]
         for key, value in reaction.get("inputs", {}).items()
         for c in value.get("components", [])
         if c.get("reaction_role") == "REACTANT"]
    )
    
    temperature = reaction.get("conditions", {}).get("temperature", {}).get("setpoint", {}).get("value", None)
    products = " .".join(
        [p["identifiers"][0]["value"]
         for outcome in reaction.get("outcomes", [])
         for p in outcome.get("products", [])]
    )
    
    yield_percentage = next(
        (m.get("percentage", {}).get("value")
         for outcome in reaction.get("outcomes", [])
         for p in outcome.get("products", [])
         for m in p.get("measurements", [])
         if m.get("type") == "YIELD"),
        None
    )
    
    if reactants and products and yield_percentage is not None:
        reaction_smiles = f"{reactants}>>{products}"
        rows.append([reaction_smiles, temperature, yield_percentage])

df = pd.DataFrame(rows, columns=["reaction_smiles", "temperature_C", "yield_percentage"])

df["temperature_C"] = pd.to_numeric(df["temperature_C"], errors="coerce")
df["temperature_C"].fillna(df["temperature_C"].median(), inplace=True)

df["yield_percentage"] = pd.to_numeric(df["yield_percentage"], errors="coerce")
df["yield_percentage"] = df["yield_percentage"].astype(float) / 100

train_smiles, test_smiles, train_y, test_y = train_test_split(
    df["reaction_smiles"], df["yield_percentage"], test_size=0.2, random_state=42
)

class ChemDataset(Dataset):
    def __init__(self, smiles, labels, tokenizer):
        self.smiles = smiles.tolist() if isinstance(smiles, pd.Series) else smiles
        self.labels = labels.tolist() if isinstance(labels, pd.Series) else labels
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.smiles)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.smiles[idx],
            truncation=True,
            padding="max_length",
            max_length=128,
            return_tensors="pt"
        )
        encoding = {key: val.squeeze(0) for key, val in encoding.items()}
        encoding["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        return encoding

model_name = "DeepChem/ChemBERTa-77M-MLM"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=1)

train_dataset = ChemDataset(train_smiles, train_y, tokenizer)
test_dataset = ChemDataset(test_smiles, test_y, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
loss_fn = torch.nn.MSELoss()

epochs = 5
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        inputs = {key: val.to(device) for key, val in batch.items() if key != "labels"}
        labels = batch["labels"].to(device).unsqueeze(1)
        outputs = model(**inputs).logits
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss / len(train_loader):.4f}")

model.eval()
preds, true_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        inputs = {key: val.to(device) for key, val in batch.items() if key != "labels"}
        labels = batch["labels"].to(device)
        outputs = model(**inputs).logits.squeeze().cpu().numpy()
        preds.extend(outputs)
        true_labels.extend(labels.cpu().numpy())

mse = mean_squared_error(true_labels, preds)
mae = mean_absolute_error(true_labels, preds)
r2 = r2_score(true_labels, preds)

print(f"Final MSE: {mse:.4f}")
print(f"Final MAE: {mae:.4f}")
print(f"Final R² Score: {r2:.4f}")

/var/folders/_4/6dhlypk978vdpg6_glxd3wpc0000gn/T/ipykernel_3336/2941092405.py:45: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["temperature_C"].fillna(df["temperature_C"].median(), inplace=True)
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at DeepChem/ChemBERTa-77M-MLM and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able

Epoch 1/5, Loss: 0.1066
Epoch 2/5, Loss: 0.0861
Epoch 3/5, Loss: 0.0840
Epoch 4/5, Loss: 0.0801
Epoch 5/5, Loss: 0.0758
Final MSE: 0.0717
Final MAE: 0.2139
Final R² Score: 0.1078
Training Complete.
